
# DiT Family — Reference Notebook

This is the companion notebook to `ViT_Models_Reference.ipynb`, covering every model in your
"DiT — Priority List." Same structure: **explanation → from-scratch minimal implementation →
tiny demo forward pass**, for each model.

## Roadmap covered here

```
01. DiT                -> Transformer as a diffusion denoiser (replaces the U-Net)
02. PixArt-alpha/Sigma  -> Efficient T2I DiT with cross-attention to text
03. MMDiT               -> Joint dual-stream text+image Transformer (used in SD3)
04. SD3 / SD3.5         -> MMDiT backbone + flow matching training objective
05. FLUX                -> Hybrid parallel-block MMDiT-style T2I model
06. SiT                 -> Flow/score-based reformulation of DiT
07. U-ViT                -> U-Net-style long skip connections + Transformer blocks
08. Video DiT (general) -> Spatiotemporal patchify + attention for video latents
09. Wan / HunyuanVideo / CogVideoX / LTX-Video -> production video-DiT families
```

**"Ones to actually build from scratch":** DiT → PixArt → MMDiT → FLUX → SiT/Flow-Matching →
Video DiT. This notebook implements all of those in full, and gives concise explanations +
architectural diagrams-in-code-comments for everything else in the table.

**Key throughline:** almost everything here is "take the DiT block, then change (a) *what*
it's conditioned on, (b) *how* text and image tokens interact, and/or (c) *what loss* trains
it (epsilon-prediction diffusion vs. flow matching)." Once you have DiT + MMDiT + flow
matching, you can read the architecture diagram of almost any modern generative Transformer.

> Note: run this in an environment with `torch` installed (`pip install torch`).


In [1]:

# If needed:
# !pip install torch --quiet

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda



## 1. DiT (Diffusion Transformer) — ⭐⭐⭐⭐⭐ Foundation

**Idea:** diffusion models were originally denoised by a U-Net (conv + attention). DiT
replaces the U-Net entirely with a **plain ViT-style Transformer** that operates on
patchified latents (the compressed representation from a VAE, not raw pixels). Conditioning
information (the diffusion timestep `t`, and optionally a class label) is injected via
**adaptive LayerNorm (adaLN-Zero)**: instead of normal LayerNorm's fixed learned scale/shift,
the scale, shift, *and* a residual gate are **predicted from the conditioning embedding** at
every block, and the gate is zero-initialized so each block starts as an identity function
(this makes very deep conditioned Transformers stable to train from scratch).

**Why it matters:** proved Transformers scale better than U-Nets for diffusion (the
"scaling laws for diffusion" result) — this is *the* architectural ancestor of essentially
every serious modern image/video diffusion model.

**Used today for:** the backbone pattern for FLUX, SD3, PixArt, and nearly all video
diffusion models below.


In [2]:

def sinusoidal_timestep_embedding(t, dim, max_period=10000):
    """Standard transformer-style sinusoidal embedding of a scalar diffusion timestep."""
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device).float() / half)
    args = t[:, None].float() * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        emb = F.pad(emb, (0, 1))
    return emb


class TimestepEmbedder(nn.Module):
    def __init__(self, dim, freq_dim=256):
        super().__init__()
        self.freq_dim = freq_dim
        self.mlp = nn.Sequential(nn.Linear(freq_dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        freqs = sinusoidal_timestep_embedding(t, self.freq_dim)
        return self.mlp(freqs)


def modulate(x, shift, scale):
    """adaLN modulation: scale/shift a normalized tensor, conditioned on `t` (and class)."""
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class MHSA(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1) * self.scale).softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(out)


class MLP(nn.Module):
    def __init__(self, dim, hidden_ratio=4.0):
        super().__init__()
        hidden = int(dim * hidden_ratio)
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(approximate="tanh"), nn.Linear(hidden, dim))

    def forward(self, x):
        return self.net(x)


class DiTBlock(nn.Module):
    """adaLN-Zero conditioned Transformer block -- the core reusable unit of the whole DiT family."""
    def __init__(self, dim, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.attn = MHSA(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        self.mlp = MLP(dim, mlp_ratio)
        # predicts 6 modulation params from the conditioning vector: shift/scale/gate x2 (attn, mlp)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.adaLN_modulation[-1].weight)   # zero-init -> block starts as identity
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x, cond):
        shift_a, scale_a, gate_a, shift_m, scale_m, gate_m = self.adaLN_modulation(cond).chunk(6, dim=-1)
        x = x + gate_a.unsqueeze(1) * self.attn(modulate(self.norm1(x), shift_a, scale_a))
        x = x + gate_m.unsqueeze(1) * self.mlp(modulate(self.norm2(x), shift_m, scale_m))
        return x


class FinalLayer(nn.Module):
    """adaLN-modulated final projection back to patch pixels/latent-channels."""
    def __init__(self, dim, patch_size, out_channels):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.linear = nn.Linear(dim, patch_size * patch_size * out_channels)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x, cond):
        shift, scale = self.adaLN_modulation(cond).chunk(2, dim=-1)
        x = modulate(self.norm(x), shift, scale)
        return self.linear(x)


class DiT(nn.Module):
    """
    Operates on a latent tensor (B, in_channels, H, W) -- e.g. the output of a VAE encoder,
    NOT raw pixels. Predicts noise (or velocity, see SD3/SiT sections) of the same shape.
    """
    def __init__(self, input_size=32, patch_size=2, in_channels=4, dim=192, depth=6,
                 num_heads=3, num_classes=10):
        super().__init__()
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.grid_size = input_size // patch_size
        num_patches = self.grid_size ** 2

        self.x_embed = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.t_embed = TimestepEmbedder(dim)
        self.y_embed = nn.Embedding(num_classes + 1, dim)   # +1 slot for "unconditional" (classifier-free guidance)

        self.blocks = nn.ModuleList([DiTBlock(dim, num_heads) for _ in range(depth)])
        self.final_layer = FinalLayer(dim, patch_size, in_channels)

    def unpatchify(self, x):
        B, N, _ = x.shape
        p, c, g = self.patch_size, self.in_channels, self.grid_size
        x = x.reshape(B, g, g, p, p, c)
        x = x.permute(0, 5, 1, 3, 2, 4).reshape(B, c, g * p, g * p)
        return x

    def forward(self, x, t, y):
        x = self.x_embed(x).flatten(2).transpose(1, 2) + self.pos_embed   # (B, N, dim)
        cond = self.t_embed(t) + self.y_embed(y)                          # fuse timestep + class into ONE vector
        for blk in self.blocks:
            x = blk(x, cond)
        x = self.final_layer(x, cond)                                     # (B, N, patch*patch*C)
        return self.unpatchify(x)                                         # (B, C, H, W) predicted noise


# --- demo: one denoising forward pass on fake VAE-latent-sized input ---
dit = DiT(input_size=32, patch_size=2, in_channels=4, dim=192, depth=4, num_heads=3, num_classes=10)
noisy_latents = torch.randn(2, 4, 32, 32)
t = torch.randint(0, 1000, (2,))
y = torch.randint(0, 10, (2,))
pred_noise = dit(noisy_latents, t, y)
print("DiT predicted noise shape:", pred_noise.shape)


DiT predicted noise shape: torch.Size([2, 4, 32, 32])



## 2. PixArt-α / PixArt-Σ — ⭐⭐⭐⭐ Efficient text-to-image DiT

**Idea:** take the DiT block and add **cross-attention to text token embeddings** (from a
frozen pretrained text encoder like T5), inserted between the self-attention and MLP
sub-layers of every block. Class-label conditioning (DiT) is replaced by full free-form text
conditioning. PixArt's headline contribution is *efficiency*: careful architecture and
training-recipe choices (decomposing training into stages: pixel-dependency learning →
text-image alignment → high-resolution/aesthetic fine-tuning) let it reach FLUX/SD3-adjacent
quality at a fraction of the training compute. PixArt-Σ scales this to higher resolution and
adds a stronger, longer-context text encoder.

**Why it matters:** demonstrated that competitive T2I quality doesn't require the compute
budget of the largest labs — the training *recipe* matters as much as scale.

**Used today for:** efficient/open text-to-image generation.


In [3]:

class CrossAttention(nn.Module):
    """Image (query) tokens attend into text (key/value) tokens."""
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim * 2)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, context):
        B, N, C = x.shape
        M = context.shape[1]
        q = self.q(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        kv = self.kv(context).reshape(B, M, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1) * self.scale).softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(out)


class PixArtBlock(nn.Module):
    """DiTBlock + a cross-attention sub-layer to a frozen text encoder's token embeddings."""
    def __init__(self, dim, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.self_attn = MHSA(dim, num_heads)
        self.cross_attn = CrossAttention(dim, num_heads)          # <-- the new piece vs. plain DiT
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        self.mlp = MLP(dim, mlp_ratio)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x, cond, text_tokens):
        shift_a, scale_a, gate_a, shift_m, scale_m, gate_m = self.adaLN_modulation(cond).chunk(6, dim=-1)
        x = x + gate_a.unsqueeze(1) * self.self_attn(modulate(self.norm1(x), shift_a, scale_a))
        x = x + self.cross_attn(x, text_tokens)                    # plain residual cross-attn into text
        x = x + gate_m.unsqueeze(1) * self.mlp(modulate(self.norm2(x), shift_m, scale_m))
        return x


class PixArt(nn.Module):
    def __init__(self, input_size=32, patch_size=2, in_channels=4, dim=192, depth=4,
                 num_heads=3, text_dim=128):
        super().__init__()
        self.patch_size, self.in_channels = patch_size, in_channels
        self.grid_size = input_size // patch_size
        num_patches = self.grid_size ** 2
        self.x_embed = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.t_embed = TimestepEmbedder(dim)
        self.text_proj = nn.Linear(text_dim, dim)     # projects FROZEN T5-style text embeds into model dim
        self.blocks = nn.ModuleList([PixArtBlock(dim, num_heads) for _ in range(depth)])
        self.final_layer = FinalLayer(dim, patch_size, in_channels)

    def unpatchify(self, x):
        B, N, _ = x.shape
        p, c, g = self.patch_size, self.in_channels, self.grid_size
        x = x.reshape(B, g, g, p, p, c).permute(0, 5, 1, 3, 2, 4).reshape(B, c, g * p, g * p)
        return x

    def forward(self, x, t, text_embeds):
        x = self.x_embed(x).flatten(2).transpose(1, 2) + self.pos_embed
        cond = self.t_embed(t)                          # conditioning vector for adaLN = timestep ONLY
        text_tokens = self.text_proj(text_embeds)        # (B, num_text_tokens, dim), attended via cross-attn
        for blk in self.blocks:
            x = blk(x, cond, text_tokens)
        x = self.final_layer(x, cond)
        return self.unpatchify(x)


# --- demo ---
pixart = PixArt(input_size=32, patch_size=2, in_channels=4, dim=192, depth=3, num_heads=3, text_dim=128)
noisy_latents = torch.randn(2, 4, 32, 32)
t = torch.randint(0, 1000, (2,))
frozen_text_embeds = torch.randn(2, 20, 128)   # e.g. 20 T5 tokens, pre-computed by a frozen text encoder
pred = pixart(noisy_latents, t, frozen_text_embeds)
print("PixArt predicted noise shape:", pred.shape)


PixArt predicted noise shape: torch.Size([2, 4, 32, 32])



## 3. MMDiT (Multimodal DiT) — ⭐⭐⭐⭐⭐ Joint text + image generation

**Idea:** PixArt bolts text on via cross-attention, treating text as a fixed side-input.
MMDiT (introduced for Stable Diffusion 3) instead treats **image and text tokens as two
first-class modalities that co-evolve**: each has its **own set of weights** (its own
LayerNorm, its own QKV projections, its own MLP — a "dual stream"), but at the attention
step the two streams' tokens are **concatenated and attend jointly** — so image tokens can
attend to text tokens *and* text tokens can attend to (and be updated by) image tokens, in
the same operation. This is a strictly more expressive form of conditioning than one-way
cross-attention.

**Why it matters:** lets text representations *update* over the course of generation instead
of staying frozen/static, and is the shared backbone pattern behind SD3, FLUX, and several
modern video models below.

**Used today for:** modern T2I (SD3, FLUX) — see it as "the upgrade path from PixArt-style
one-way cross-attention."


In [4]:

class MMDiTBlock(nn.Module):
    """
    Two separate token streams (image, text), each with its OWN adaLN + QKV + MLP weights,
    but a single SHARED joint self-attention operation over the concatenation of both streams.
    """
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        # image stream's own weights
        self.img_norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.img_qkv = nn.Linear(dim, dim * 3)
        self.img_proj = nn.Linear(dim, dim)
        self.img_norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        self.img_mlp = MLP(dim)
        self.img_adaLN = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))

        # text stream's own (separate!) weights
        self.txt_norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.txt_qkv = nn.Linear(dim, dim * 3)
        self.txt_proj = nn.Linear(dim, dim)
        self.txt_norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        self.txt_mlp = MLP(dim)
        self.txt_adaLN = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))

        for m in [self.img_adaLN, self.txt_adaLN]:
            nn.init.zeros_(m[-1].weight); nn.init.zeros_(m[-1].bias)

    def _qkv(self, x, qkv_layer):
        B, N, C = x.shape
        qkv = qkv_layer(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        return qkv[0], qkv[1], qkv[2]

    def forward(self, img, txt, cond):
        i_sa, i_sc, i_ga, i_sm, i_scm, i_gm = self.img_adaLN(cond).chunk(6, dim=-1)
        t_sa, t_sc, t_ga, t_sm, t_scm, t_gm = self.txt_adaLN(cond).chunk(6, dim=-1)

        img_n = modulate(self.img_norm1(img), i_sa, i_sc)
        txt_n = modulate(self.txt_norm1(txt), t_sa, t_sc)

        q_i, k_i, v_i = self._qkv(img_n, self.img_qkv)
        q_t, k_t, v_t = self._qkv(txt_n, self.txt_qkv)

        # *** the key MMDiT step: concatenate both streams and run ONE joint attention ***
        q = torch.cat([q_i, q_t], dim=2)
        k = torch.cat([k_i, k_t], dim=2)
        v = torch.cat([v_i, v_t], dim=2)
        attn = (q @ k.transpose(-2, -1) * self.scale).softmax(dim=-1)
        out = attn @ v                                            # (B, heads, N_img+N_txt, head_dim)

        n_img = img.shape[1]
        out_img, out_txt = out[:, :, :n_img], out[:, :, n_img:]
        out_img = out_img.transpose(1, 2).reshape(img.shape)
        out_txt = out_txt.transpose(1, 2).reshape(txt.shape)

        img = img + i_ga.unsqueeze(1) * self.img_proj(out_img)     # each stream applies its OWN output proj
        txt = txt + t_ga.unsqueeze(1) * self.txt_proj(out_txt)

        img = img + i_gm.unsqueeze(1) * self.img_mlp(modulate(self.img_norm2(img), i_sm, i_scm))
        txt = txt + t_gm.unsqueeze(1) * self.txt_mlp(modulate(self.txt_norm2(txt), t_sm, t_scm))
        return img, txt


# --- demo ---
dim = 192
mmdit_block = MMDiTBlock(dim, num_heads=3)
img_tokens = torch.randn(2, 16 * 16, dim)   # e.g. 16x16 patchified latent
txt_tokens = torch.randn(2, 20, dim)        # 20 text tokens
cond = torch.randn(2, dim)                  # e.g. timestep embedding (+ pooled text, in real SD3)
img_out, txt_out = mmdit_block(img_tokens, txt_tokens, cond)
print("MMDiT image stream out:", img_out.shape, "| text stream out:", txt_out.shape)


MMDiT image stream out: torch.Size([2, 256, 192]) | text stream out: torch.Size([2, 20, 192])



## 4. SD3 / SD3.5 (Stable Diffusion 3) — ⭐⭐⭐⭐⭐ MMDiT + flow matching

**Idea:** SD3 = **MMDiT backbone** (above) trained with a **flow matching** objective instead
of classic epsilon-prediction diffusion. Flow matching trains the network to directly predict
a **velocity field** that transports noise to data along a straight-line ("rectified flow")
interpolation path: `x_t = (1 - t) * x_0 + t * x_1` where `x_0` is noise and `x_1` is data,
and the target is simply the constant velocity `x_1 - x_0`. This is simpler than the
noise-schedule machinery of DDPM-style diffusion and empirically gives faster, higher-quality
sampling with fewer steps.

**Why it matters:** the pairing (MMDiT architecture + flow matching objective) is the
foundation FLUX also builds on — architecture and training-objective are separable choices,
and this is the combination that became the modern default for T2I.

**Used today for:** current-generation text-to-image / image-editing models.


In [5]:

def flow_matching_loss(model, x1, text_tokens, cond_extra_fn):
    """
    x1: clean data latents (B, C, H, W). x0: noise, same shape.
    Interpolate along the straight path x_t = (1-t) x0 + t x1, target velocity = x1 - x0.
    `model` here is expected to return a PREDICTED VELOCITY (not noise) given (x_t, t, text).
    """
    B = x1.shape[0]
    x0 = torch.randn_like(x1)
    t = torch.rand(B, device=x1.device)                      # continuous t in [0, 1], NOT a discrete timestep index
    t_ = t.view(B, 1, 1, 1)
    x_t = (1 - t_) * x0 + t_ * x1
    target_velocity = x1 - x0

    pred_velocity = model(x_t, t, text_tokens, cond_extra_fn)
    return F.mse_loss(pred_velocity, target_velocity)


class SD3Model(nn.Module):
    """Minimal MMDiT stack + patchify/unpatchify, predicting a velocity field for flow matching."""
    def __init__(self, input_size=32, patch_size=2, in_channels=4, dim=192, depth=3, num_heads=3):
        super().__init__()
        self.patch_size, self.in_channels = patch_size, in_channels
        self.grid_size = input_size // patch_size
        num_patches = self.grid_size ** 2
        self.x_embed = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.t_embed = TimestepEmbedder(dim)
        self.blocks = nn.ModuleList([MMDiTBlock(dim, num_heads) for _ in range(depth)])
        self.final_layer = FinalLayer(dim, patch_size, in_channels)

    def unpatchify(self, x):
        B, N, _ = x.shape
        p, c, g = self.patch_size, self.in_channels, self.grid_size
        return x.reshape(B, g, g, p, p, c).permute(0, 5, 1, 3, 2, 4).reshape(B, c, g * p, g * p)

    def forward(self, x_t, t, text_tokens, cond_extra_fn=None):
        img = self.x_embed(x_t).flatten(2).transpose(1, 2) + self.pos_embed
        cond = self.t_embed(t * 1000)     # flow-matching t in [0,1] -> reuse sinusoidal embed on a [0,1000] scale
        for blk in self.blocks:
            img, text_tokens = blk(img, text_tokens, cond)
        out = self.final_layer(img, cond)
        return self.unpatchify(out)


# --- demo: one flow-matching training step ---
sd3 = SD3Model(input_size=32, patch_size=2, in_channels=4, dim=192, depth=2, num_heads=3)
x1 = torch.randn(2, 4, 32, 32)                 # "clean" latents (stand-in for real VAE-encoded images)
text_tokens = torch.randn(2, 20, 192)
loss = flow_matching_loss(sd3, x1, text_tokens, cond_extra_fn=None)
loss.backward()
print("SD3-style flow matching loss:", loss.item())


SD3-style flow matching loss: 1.9610544443130493



## 5. FLUX — ⭐⭐⭐⭐⭐ Modern high-quality T2I

**Idea:** FLUX (Black Forest Labs) is, at a high level, in the same architecture *family* as
SD3 — an MMDiT-style dual-stream design trained with flow matching — but scaled up and with
some efficiency-oriented architectural choices that later FLUX-style models popularized:
- **Parallel attention + MLP blocks** (in the later "single-stream" blocks, after the
  dual-stream blocks): attention and MLP are computed **in parallel from the same normalized
  input** and summed, rather than sequentially — fewer serial LayerNorms, better hardware
  utilization at scale.
- **Rotary position embeddings (RoPE)** instead of learned/absolute position embeddings, for
  better length/resolution generalization.
- **Guidance distillation**: a separate distilled model absorbs classifier-free guidance so
  sampling doesn't need two forward passes per step.

FLUX's exact trained weights and some implementation details aren't public, but the
architectural *pattern* (parallel attn/MLP + RoPE + MMDiT-style joint attention, flow
matching objective) is documented and is what's implemented below.

**Used today for:** current state-of-the-art open-weight text-to-image generation and
editing.


In [6]:

def rope_freqs(seq_len, dim, base=10000.0, device="cpu"):
    """Rotary position embedding frequency table."""
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))
    pos = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(pos, inv_freq)           # (seq_len, dim/2)
    return torch.cat([freqs, freqs], dim=-1)       # (seq_len, dim)


def apply_rope(x, freqs):
    """x: (B, heads, N, head_dim). Rotates pairs of dims by the position-dependent angle."""
    def rotate_half(t):
        t1, t2 = t.chunk(2, dim=-1)
        return torch.cat([-t2, t1], dim=-1)
    cos, sin = freqs.cos(), freqs.sin()
    return x * cos + rotate_half(x) * sin


class ParallelBlock(nn.Module):
    """
    FLUX-style 'single-stream' block: attention and MLP computed IN PARALLEL from the
    same normalized+modulated input (not sequentially), then both added back to the residual.
    Uses RoPE instead of learned absolute position embeddings.
    """
    def __init__(self, dim, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.qkv = nn.Linear(dim, dim * 3)
        hidden = int(dim * mlp_ratio)
        self.mlp_in = nn.Linear(dim, hidden)
        self.act = nn.GELU(approximate="tanh")
        self.out_proj = nn.Linear(dim + hidden, dim)   # fuses attn-out and mlp-out in ONE projection
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 3 * dim))
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x, cond, rope_freqs_table):
        shift, scale, gate = self.adaLN_modulation(cond).chunk(3, dim=-1)
        x_n = modulate(self.norm(x), shift, scale)

        B, N, C = x_n.shape
        qkv = self.qkv(x_n).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = apply_rope(q, rope_freqs_table), apply_rope(k, rope_freqs_table)   # RoPE, not additive pos-embed
        attn_out = ((q @ k.transpose(-2, -1) * self.scale).softmax(-1) @ v)
        attn_out = attn_out.transpose(1, 2).reshape(B, N, C)

        mlp_out = self.act(self.mlp_in(x_n))                     # computed IN PARALLEL with attention, not after

        fused = self.out_proj(torch.cat([attn_out, mlp_out], dim=-1))
        return x + gate.unsqueeze(1) * fused


class FluxLikeModel(nn.Module):
    """Dual-stream MMDiT blocks first (text+image co-evolve), then parallel single-stream blocks."""
    def __init__(self, input_size=32, patch_size=2, in_channels=4, dim=192,
                 dual_depth=2, single_depth=2, num_heads=3):
        super().__init__()
        self.patch_size, self.in_channels = patch_size, in_channels
        self.grid_size = input_size // patch_size
        self.x_embed = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)
        self.t_embed = TimestepEmbedder(dim)
        self.dual_blocks = nn.ModuleList([MMDiTBlock(dim, num_heads) for _ in range(dual_depth)])
        self.single_blocks = nn.ModuleList([ParallelBlock(dim, num_heads) for _ in range(single_depth)])
        self.final_layer = FinalLayer(dim, patch_size, in_channels)
        self.head_dim = dim // num_heads

    def unpatchify(self, x):
        B, N, _ = x.shape
        p, c, g = self.patch_size, self.in_channels, self.grid_size
        return x.reshape(B, g, g, p, p, c).permute(0, 5, 1, 3, 2, 4).reshape(B, c, g * p, g * p)

    def forward(self, x_t, t, text_tokens):
        img = self.x_embed(x_t).flatten(2).transpose(1, 2)      # NOTE: no additive pos_embed -- RoPE handles it
        cond = self.t_embed(t * 1000)

        for blk in self.dual_blocks:                             # stage 1: text and image co-evolve
            img, text_tokens = blk(img, text_tokens, cond)

        merged = torch.cat([img, text_tokens], dim=1)             # stage 2: single combined stream
        freqs = rope_freqs(merged.shape[1], self.head_dim, device=merged.device)
        for blk in self.single_blocks:
            merged = blk(merged, cond, freqs)

        img_out = merged[:, :img.shape[1]]
        out = self.final_layer(img_out, cond)
        return self.unpatchify(out)


# --- demo ---
flux_like = FluxLikeModel(input_size=32, patch_size=2, in_channels=4, dim=192,
                           dual_depth=2, single_depth=2, num_heads=3)
x_t = torch.randn(2, 4, 32, 32)
t = torch.rand(2)
text_tokens = torch.randn(2, 20, 192)
pred_velocity = flux_like(x_t, t, text_tokens)
print("FLUX-like predicted velocity shape:", pred_velocity.shape)


FLUX-like predicted velocity shape: torch.Size([2, 4, 32, 32])



## 6. SiT — ⭐⭐⭐⭐ Diffusion/flow Transformer

**Idea:** SiT ("Scalable Interpolant Transformers") takes the *exact same* DiT architecture
and asks: what if, instead of committing to classic DDPM epsilon-prediction, we treat the
architecture and the **generative formulation** (interpolant / SDE vs. ODE, velocity vs.
noise vs. score prediction) as independent design axes? SiT shows you can plug in a general
**stochastic interpolant** framework — of which both diffusion and flow matching are special
cases — on top of an unmodified DiT backbone, and that this framework consistently
outperforms the original fixed-recipe DiT.

**Why it matters:** clarifies that "DiT vs. flow matching" isn't really "two different
models" — it's the *same Transformer backbone* with a swappable training objective. This is
the cleanest way to understand how DiT → SD3/FLUX's flow-matching objective are related.

**Used today for:** flow-matching / diffusion-Transformer research; conceptually, this is
"why FLUX/SD3's objective works," explained architecture-agnostically.


In [7]:

class SiTVelocityLoss:
    """
    SiT / stochastic-interpolant framing, showing DDPM-style epsilon-prediction and flow
    matching as two special cases of the SAME interpolant x_t = alpha(t) x1 + sigma(t) x0.
    """
    @staticmethod
    def linear_interpolant_loss(model, x1, t_embed_scale=1000.0, **model_kwargs):
        """alpha(t) = t, sigma(t) = 1-t  -> this recovers exactly the flow-matching loss."""
        B = x1.shape[0]
        x0 = torch.randn_like(x1)
        t = torch.rand(B, device=x1.device)
        t_ = t.view(B, *([1] * (x1.dim() - 1)))
        x_t = t_ * x1 + (1 - t_) * x0
        target_velocity = x1 - x0                       # d/dt [t*x1 + (1-t)*x0]
        pred = model(x_t, t, **model_kwargs)
        return F.mse_loss(pred, target_velocity)

    @staticmethod
    def variance_preserving_interpolant_loss(model, x1, t_embed_scale=1000.0, **model_kwargs):
        """alpha(t) = cos(pi/2 * t), sigma(t) = sin(pi/2 * t) -> a DDPM-like (variance-preserving) path."""
        B = x1.shape[0]
        x0 = torch.randn_like(x1)
        t = torch.rand(B, device=x1.device)
        t_ = t.view(B, *([1] * (x1.dim() - 1)))
        alpha, sigma = torch.cos(math.pi / 2 * t_), torch.sin(math.pi / 2 * t_)
        x_t = alpha * x1 + sigma * x0
        # target is the time-derivative of the interpolant (the "velocity" for THIS path)
        dalpha = -math.pi / 2 * torch.sin(math.pi / 2 * t_)
        dsigma = math.pi / 2 * torch.cos(math.pi / 2 * t_)
        target_velocity = dalpha * x1 + dsigma * x0
        pred = model(x_t, t, **model_kwargs)
        return F.mse_loss(pred, target_velocity)


# --- demo: SAME DiT backbone, two different interpolant paths / losses ---
sit_backbone = DiT(input_size=32, patch_size=2, in_channels=4, dim=192, depth=2, num_heads=3, num_classes=10)

def model_fn(x_t, t, y):
    return sit_backbone(x_t, (t * 1000).long().clamp(max=999), y)

x1 = torch.randn(2, 4, 32, 32)
y = torch.randint(0, 10, (2,))

loss_flow = SiTVelocityLoss.linear_interpolant_loss(model_fn, x1, y=y)
loss_vp = SiTVelocityLoss.variance_preserving_interpolant_loss(model_fn, x1, y=y)
print("Same backbone, linear (flow-matching-equivalent) interpolant loss:", loss_flow.item())
print("Same backbone, variance-preserving (diffusion-like) interpolant loss:", loss_vp.item())


Same backbone, linear (flow-matching-equivalent) interpolant loss: 2.0351061820983887
Same backbone, variance-preserving (diffusion-like) interpolant loss: 2.4536736011505127



## 7. U-ViT — ⭐⭐⭐⭐ U-Net ideas + Transformer

**Idea:** U-ViT keeps the U-Net's most useful structural trick — **long skip connections**
between shallow (early) and deep (late) layers, which help gradient flow and let late layers
reuse low-level information — but implements the whole network as a plain sequence of
Transformer blocks instead of convolutional down/up-sampling stages. Skip connections are
implemented by **concatenating** an early block's output tokens with a later block's input
tokens along the channel dimension, then projecting back down with a linear layer (rather
than a U-Net's spatial concatenation across resolutions).

**Why it matters:** an early demonstration (pre-dating DiT's adaLN-Zero recipe) that a
"pure Transformer, no convolutions at all" diffusion backbone works well; its skip-connection
idea is a useful architectural variant to know even where later models (DiT/MMDiT) dropped it
in favor of pure adaLN conditioning.

**Used today for:** diffusion-Transformer research; a design point between "U-Net" and
"pure DiT."


In [8]:

class UViT(nn.Module):
    """
    Symmetric Transformer with U-Net-style long skip connections: block i's output (in the
    first half) is concatenated with block (depth-1-i)'s input (in the second half).
    """
    def __init__(self, input_size=32, patch_size=2, in_channels=4, dim=192, depth=6, num_heads=3):
        super().__init__()
        assert depth % 2 == 0
        self.patch_size, self.in_channels = patch_size, in_channels
        self.grid_size = input_size // patch_size
        num_patches = self.grid_size ** 2

        self.x_embed = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.t_embed = TimestepEmbedder(dim)

        self.in_blocks = nn.ModuleList([DiTBlock(dim, num_heads) for _ in range(depth // 2)])
        self.mid_block = DiTBlock(dim, num_heads)
        self.out_blocks = nn.ModuleList([DiTBlock(dim, num_heads) for _ in range(depth // 2)])
        # each "out" block gets a skip-fuse linear that maps concat([2*dim]) -> dim before the block runs
        self.skip_fuse = nn.ModuleList([nn.Linear(2 * dim, dim) for _ in range(depth // 2)])
        self.final_layer = FinalLayer(dim, patch_size, in_channels)

    def unpatchify(self, x):
        B, N, _ = x.shape
        p, c, g = self.patch_size, self.in_channels, self.grid_size
        return x.reshape(B, g, g, p, p, c).permute(0, 5, 1, 3, 2, 4).reshape(B, c, g * p, g * p)

    def forward(self, x, t, cond_extra=None):
        h = self.x_embed(x).flatten(2).transpose(1, 2) + self.pos_embed
        cond = self.t_embed(t * 1000)
        if cond_extra is not None:
            cond = cond + cond_extra

        skips = []
        for blk in self.in_blocks:
            h = blk(h, cond)
            skips.append(h)                                    # stash for the matching "out" block

        h = self.mid_block(h, cond)

        for blk, fuse in zip(self.out_blocks, self.skip_fuse):
            skip = skips.pop()                                  # LIFO: pairs early block <-> late block
            h = fuse(torch.cat([h, skip], dim=-1))               # U-Net-style long skip, done via concat+linear
            h = blk(h, cond)

        out = self.final_layer(h, cond)
        return self.unpatchify(out)


# --- demo ---
uvit = UViT(input_size=32, patch_size=2, in_channels=4, dim=192, depth=4, num_heads=3)
x = torch.randn(2, 4, 32, 32)
t = torch.rand(2)
pred = uvit(x, t)
print("U-ViT predicted noise/velocity shape:", pred.shape)


U-ViT predicted noise/velocity shape: torch.Size([2, 4, 32, 32])



## 8. Video DiT (general pattern) — ⭐⭐⭐⭐⭐ Spatiotemporal generation

**Idea:** extend the image-DiT recipe to video latents (typically a spatio-temporally
compressed 3D VAE latent, shape `(C, T, H, W)`). The two changes needed are exactly the same
ones ViViT needed for *understanding* (see the ViT notebook): **3D/tubelet patchify** instead
of 2D patchify, and **spatiotemporal attention** — either full joint attention over every
space-time token (expensive but most expressive; what CogVideoX does) or **factorized**
space-then-time attention (cheaper; common in efficiency-oriented models). Conditioning
(timestep, text) is injected the same adaLN / cross-attention / MMDiT ways as image DiTs.

**Why it matters:** this is the shared template underneath Wan, HunyuanVideo, CogVideoX, and
LTX-Video below — they mostly differ in *which* attention factorization, *how* text is fused
(cross-attention vs. MMDiT-style joint), and training/compression efficiency choices.

**Used today for:** the backbone of essentially all current text/image-to-video models.


In [9]:

class SpatioTemporalDiTBlock(nn.Module):
    """adaLN-conditioned DiT block, but attention factorizes into a spatial pass + a temporal pass."""
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm_s = nn.LayerNorm(dim, elementwise_affine=False)
        self.spatial_attn = MHSA(dim, num_heads)
        self.norm_t = nn.LayerNorm(dim, elementwise_affine=False)
        self.temporal_attn = MHSA(dim, num_heads)
        self.norm_mlp = nn.LayerNorm(dim, elementwise_affine=False)
        self.mlp = MLP(dim)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 9 * dim))  # 3 sub-layers x (shift,scale,gate)
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x, cond, thw):
        T, H, W = thw
        B, N, D = x.shape
        (sh_s, sc_s, g_s, sh_t, sc_t, g_t, sh_m, sc_m, g_m) = self.adaLN_modulation(cond).chunk(9, dim=-1)

        x_sp = modulate(self.norm_s(x), sh_s, sc_s).view(B * T, H * W, D)
        x_sp = self.spatial_attn(x_sp).view(B, T, H * W, D).reshape(B, N, D)
        x = x + g_s.unsqueeze(1) * x_sp

        x_t = modulate(self.norm_t(x), sh_t, sc_t).view(B, T, H * W, D).permute(0, 2, 1, 3).reshape(B * H * W, T, D)
        x_t = self.temporal_attn(x_t).view(B, H * W, T, D).permute(0, 2, 1, 3).reshape(B, N, D)
        x = x + g_t.unsqueeze(1) * x_t

        x = x + g_m.unsqueeze(1) * self.mlp(modulate(self.norm_mlp(x), sh_m, sc_m))
        return x


class VideoDiT(nn.Module):
    def __init__(self, tubelet_size=(1, 2, 2), in_channels=4, dim=192, depth=3, num_heads=3):
        super().__init__()
        self.tubelet_embed = nn.Conv3d(in_channels, dim, kernel_size=tubelet_size, stride=tubelet_size)
        self.in_channels = in_channels
        self.tubelet_size = tubelet_size
        self.t_embed = TimestepEmbedder(dim)
        self.blocks = nn.ModuleList([SpatioTemporalDiTBlock(dim, num_heads) for _ in range(depth)])
        self.final_layer = FinalLayer(dim, patch_size=1, out_channels=0)  # placeholder, replaced below
        # final projection back to per-tubelet latent values
        self.out_proj = nn.Linear(dim, tubelet_size[0] * tubelet_size[1] * tubelet_size[2] * in_channels)
        self.out_norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.out_adaLN = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        nn.init.zeros_(self.out_adaLN[-1].weight); nn.init.zeros_(self.out_adaLN[-1].bias)
        nn.init.zeros_(self.out_proj.weight); nn.init.zeros_(self.out_proj.bias)

    def forward(self, x, t):                                  # x: (B, C, T, H, W) video latent
        B, C, T, H, W = x.shape
        tt, th, tw = self.tubelet_size
        h_out = self.tubelet_embed(x)                          # (B, dim, T', H', W')
        _, D, T2, H2, W2 = h_out.shape
        tokens = h_out.flatten(2).transpose(1, 2)               # (B, T'*H'*W', dim)

        cond = self.t_embed(t * 1000)
        for blk in self.blocks:
            tokens = blk(tokens, cond, (T2, H2, W2))

        shift, scale = self.out_adaLN(cond).chunk(2, dim=-1)
        tokens = modulate(self.out_norm(tokens), shift, scale)
        tokens = self.out_proj(tokens)                          # (B, N, tt*th*tw*C)

        out = tokens.view(B, T2, H2, W2, tt, th, tw, C)
        out = out.permute(0, 7, 1, 4, 2, 5, 3, 6).reshape(B, C, T2 * tt, H2 * th, W2 * tw)
        return out


# --- demo ---
video_dit = VideoDiT(tubelet_size=(1, 2, 2), in_channels=4, dim=192, depth=2, num_heads=3)
video_latent = torch.randn(1, 4, 8, 16, 16)   # (B, C, T, H, W) -- e.g. an 8-frame compressed video latent
t = torch.rand(1)
pred = video_dit(video_latent, t)
print("Video DiT predicted noise/velocity shape:", pred.shape)


Video DiT predicted noise/velocity shape: torch.Size([1, 4, 8, 16, 16])


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))



## 9. Wan, HunyuanVideo, CogVideoX, LTX-Video — ⭐⭐⭐⭐ Production video-DiT families

These four are all **instances of the "Video DiT" pattern above** — same core idea
(3D-patchify latents + spatiotemporal attention + timestep/text conditioning, trained with a
flow-matching-style objective), differing mainly in scale, attention factorization, and
text-fusion strategy. Rather than re-implementing four near-duplicate architectures, here's
what's actually distinct about each, in the terms the code above already introduced:

- **Wan** — open-weight text/image-to-video model family. Uses a Video-DiT backbone with
  **cross-attention text conditioning** (PixArt-style, rather than MMDiT dual-stream) and a
  flow-matching objective; notable for strong open-weight I2V (image-to-video, i.e.
  conditioning generation on a starting frame in addition to text) quality.
- **HunyuanVideo** — large-scale T2V/I2V model using a **dual-stream-then-single-stream**
  design directly analogous to the FLUX pattern above (MMDiT-style joint text/video blocks,
  followed by parallel single-stream blocks), scaled to video tubelets instead of image
  patches.
- **CogVideoX** — uses **full 3D joint attention** (not factorized space/time — every
  space-time token attends to every other one at once, like swapping the
  `SpatioTemporalDiTBlock`'s two-pass factorized attention above for one single
  `MHSA` call over all `T*H*W` tokens) plus an "expert" adaptive-LayerNorm variant that gives
  text and video tokens separate adaLN parameters despite sharing attention — a middle ground
  between PixArt's cross-attention and MMDiT's fully separate dual streams.
- **LTX-Video** — optimized for **speed**: a very high-compression video VAE (so there are far
  fewer video tokens for the DiT to process in the first place) paired with a comparatively
  lightweight Video-DiT, targeting fast/interactive generation rather than maximum quality.

**The one code change that separates them**, in terms of what's already implemented above:
`SpatioTemporalDiTBlock`'s factorized (space-pass, time-pass) attention *is* the efficient
end of this spectrum; replacing its two attention calls with a single `MHSA` over all
`T*H*W` tokens at once *is* CogVideoX's full-3D-attention end of the spectrum. Wan sits
between PixArt and this block (swap `SpatioTemporalDiTBlock`'s self-attention-only design for
the `PixArtBlock` idea, factorized over space-time). HunyuanVideo is the video-tubelet version
of `FluxLikeModel`.


In [10]:

class FullJointAttentionVideoBlock(nn.Module):
    """
    CogVideoX-style: ONE joint attention over every space-time token together (no space/time
    factorization split), contrasted directly with SpatioTemporalDiTBlock above.
    """
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.attn = MHSA(dim, num_heads)                 # single full attention over ALL T*H*W tokens
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        self.mlp = MLP(dim)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x, cond):
        shift_a, scale_a, gate_a, shift_m, scale_m, gate_m = self.adaLN_modulation(cond).chunk(6, dim=-1)
        x = x + gate_a.unsqueeze(1) * self.attn(modulate(self.norm1(x), shift_a, scale_a))
        x = x + gate_m.unsqueeze(1) * self.mlp(modulate(self.norm2(x), shift_m, scale_m))
        return x


# --- demo: same tubelet embedding, but full joint space-time attention instead of factorized ---
tubelet_embed = nn.Conv3d(4, 192, kernel_size=(1, 2, 2), stride=(1, 2, 2))
video_latent = torch.randn(1, 4, 8, 16, 16)
tokens = tubelet_embed(video_latent).flatten(2).transpose(1, 2)   # (B, T*H*W, dim) -- all tokens flattened together
print("tokens fed into ONE joint attention call (CogVideoX-style):", tokens.shape)

block = FullJointAttentionVideoBlock(dim=192, num_heads=3)
cond = torch.randn(1, 192)
out = block(tokens, cond)
print("output:", out.shape, "-- compare compute cost to the two-pass factorized block above")


tokens fed into ONE joint attention call (CogVideoX-style): torch.Size([1, 512, 192])
output: torch.Size([1, 512, 192]) -- compare compute cost to the two-pass factorized block above



## Summary — how these models actually relate to each other

| Model | Built on | Core new idea | Where it shows up again |
|---|---|---|---|
| **DiT** | ViT | adaLN-Zero conditioned Transformer as diffusion denoiser | every model below |
| **PixArt-α/Σ** | DiT | cross-attention to frozen text-encoder tokens | efficient T2I |
| **MMDiT** | DiT | dual-stream image+text, joint shared attention | SD3, FLUX, HunyuanVideo |
| **SD3/3.5** | MMDiT | + flow matching training objective | modern T2I default recipe |
| **FLUX** | MMDiT | + parallel attn/MLP blocks, RoPE, guidance distillation | current SOTA open T2I |
| **SiT** | DiT | generalizes objective: interpolants unify diffusion & flow matching | clarifies why SD3/FLUX's loss works |
| **U-ViT** | DiT-adjacent | U-Net-style long skip connections in a pure Transformer | historical/alternative design point |
| **Video DiT** | DiT | 3D tubelet patchify + spatiotemporal attention | Wan, HunyuanVideo, CogVideoX, LTX-Video |
| **Wan** | Video DiT + PixArt-style conditioning | open T2V/I2V | video generation |
| **HunyuanVideo** | Video DiT + MMDiT/FLUX pattern | large-scale T2V/I2V | video generation |
| **CogVideoX** | Video DiT | full joint 3D attention (no factorization) | video generation |
| **LTX-Video** | Video DiT | heavy VAE compression for speed | fast/real-time-ish video generation |

**Reading order if revisiting:** DiT → PixArt → MMDiT → SD3 → FLUX → SiT → U-ViT → Video DiT →
(Wan / HunyuanVideo / CogVideoX / LTX-Video as variations on Video DiT). Once DiT + MMDiT +
flow matching click, the video models are "the same ideas, in 3D."

**Cross-reference to the ViT notebook:** DiT's patchify step *is* ViT's `PatchEmbed`; Video
DiT's tubelet embedding *is* ViViT's `TubeletEmbed`; and the flow of "self-attention block ->
add conditioning" throughout this notebook reuses the exact `MHSA`/`MLP` building blocks from
the ViT notebook's section 1 — these two notebooks share one underlying Transformer-block
vocabulary, applied to understanding (ViT) vs. generation (DiT).
